<a href="https://github.com/gutris1/segsmaker">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a>

---
> **Segsmaker — Optimized Fork** · Philosophy: *Speed is a feature. Efficiency is the standard.*
>
> Run cells **top-to-bottom** on first use. On subsequent sessions, skip to **Launch** directly.

In [ ]:
# @title ⚙️ **Bootstrap** — Run once per session (fetches startup helpers)
!curl -sLo ~/.conda/default.py https://github.com/gutris1/segsmaker/raw/main/script/SM/default.py && python ~/.conda/default.py

## 🐍 Conda Environment

In [ ]:
# @title 🐍 **Conda Setup** — Installs the default conda environment
# @markdown This only needs to run **once**. Skip on subsequent sessions.
!curl -sLo ~/.conda/install.py https://github.com/gutris1/segsmaker/raw/main/script/SM/conda.py
%run ~/.conda/install.py

## 🖥️ WebUI Installer

In [ ]:
# @title 🖥️ **WebUI Installer** {"display-mode":"form"}
# @markdown ### Choose your WebUI and credentials
# @markdown ---
# @markdown **Step 1 — Pick your WebUI:**
Webui = 'A1111' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
# @markdown
# @markdown **Step 2 — API Keys** *(required for Civitai downloads)*
# @markdown > 🔑 Get your Civitai key → https://civitai.com/user/account
Civitai_Key = '' # @param {type:"string", placeholder:"Paste your Civitai API key here"}
# @markdown > 🤗 Get your HF token → https://huggingface.co/settings/tokens
HF_Read_Token = '' # @param {type:"string", placeholder:"Huggingface READ token (optional but recommended)"}

import subprocess, sys
subprocess.run(['curl', '-sLo', str(Path.home() / '.conda/setup.py'),
                'https://github.com/gutris1/segsmaker/raw/main/script/SM/setup.py'], check=True)

from pathlib import Path
%run ~/.conda/setup.py

## 📥 Model Downloader

<span style="font-size:13px;">
Fill in the slots below. <b>Empty slots are automatically skipped.</b><br>
Supported sources: <code>civitai.com</code>, <code>huggingface.co</code>, direct URLs, Google Drive.<br>
Models marked <b>Persistent</b> survive session restarts. <b>Temporary</b> models are cleared on session end.
</span>

In [ ]:
# @title 📥 **Model Downloader** — 5 Checkpoint + 5 Lora + 1 VAE slots {"display-mode":"form"}
# @markdown ---
# @markdown ### 🗃️ Checkpoints *(Persistent — saved to your model folder)*
# @markdown > Paste a full URL. Examples:
# @markdown > - `https://civitai.com/api/download/models/357609` ← direct download link
# @markdown > - `https://civitai.com/models/133005` ← model page (latest version auto-selected)
# @markdown > - `https://huggingface.co/pantat88/back_up/resolve/main/bigblu25dmix25DStyle_v10.safetensors`
Checkpoint_1 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_2 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_3 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_4 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_5 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### 🎨 LoRA *(Persistent — saved to your Lora folder)*
# @markdown > Examples:
# @markdown > - `https://civitai.com/models/122359` ← Detail Tweaker XL
# @markdown > - `https://civitai.com/models/669571` ← Pony Add More Details
# @markdown > - `https://huggingface.co/Linaqruf/style-enhancer-xl-lora/resolve/main/style-enhancer-xl.safetensors`
Lora_1 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_2 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_3 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_4 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_5 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### 🎛️ VAE *(Persistent — 1 slot)*
# @markdown > Example: `https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors`
VAE_URL = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### ⚡ Speed Options
# @markdown > **Parallel Mode**: downloads all files simultaneously instead of one-by-one.
# @markdown > Recommended: **ON**. Turn off only if you experience errors.
Parallel_Download = True # @param {type:"boolean"}
Max_Workers = 3 # @param {type:"slider", min:1, max:6, step:1}
# @markdown > `Max_Workers` = how many files to download at the same time.
# @markdown > **3** is optimal for SageMaker. Use **2** on free Colab.

# ─── Build download queue (skip empty slots) ─────────────────────────────────
from nenen88 import parallel_batch_download

_ckpts = [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]
_loras = [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]

_queue = []

for _url in _ckpts:
    if _url.strip():
        _queue.append((_url.strip(), str(CKPT), None))

for _url in _loras:
    if _url.strip():
        _queue.append((_url.strip(), str(LORA), None))

if VAE_URL.strip():
    _queue.append((VAE_URL.strip(), str(VAE), None))

if not _queue:
    print('  No URLs provided — skipping download.')
elif Parallel_Download:
    parallel_batch_download(_queue, max_workers=Max_Workers)
else:
    # Sequential fallback
    for _url, _dest, _fn in _queue:
        %cd -q $_dest
        %download $_url

## 🛠️ Extra Assets *(Optional)*
<span style="font-size:13px;">
Extensions / Custom Nodes, Embeddings, Upscalers, ControlNet models.<br>
Fill in what you need and leave the rest empty.
</span>

In [ ]:
# @title 🛠️ **Extra Assets** {"display-mode":"form"}
# @markdown ---
# @markdown ### 🔌 Extension / Custom Node (git clone URL)
# @markdown > Example: `https://github.com/Mikubill/sd-webui-controlnet`
Extension_1 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_2 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_3 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
# @markdown ---
# @markdown ### 🖼️ Embeddings
# @markdown > Example: `https://huggingface.co/datasets/Nerfgun3/bad_prompt/resolve/main/bad_prompt_version2.pt`
Embedding_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Embedding_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
# @markdown ### 🔬 Upscalers
# @markdown > Example: `https://huggingface.co/gutris1/webui/resolve/main/misc/4x-UltraSharp.pth`
Upscaler_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}

# ─── Extensions: parallel clone ───────────────────────────────────────────────
from nenen88 import parallel_batch_download
import tempfile, os

_ext_urls = [u.strip() for u in [Extension_1, Extension_2, Extension_3] if u.strip()]
if _ext_urls:
    _tmp_ext = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False)
    _tmp_ext.write('\n'.join(_ext_urls))
    _tmp_ext.flush()
    _tmp_ext.close()
    print('\n⚡ Cloning extensions in parallel...')
    %cd -q $Extensions
    %clone {_tmp_ext.name}
    os.unlink(_tmp_ext.name)
# ─── Embeddings + Upscalers: parallel download ────────────────────────────────
_asset_queue = []
for _url in [Embedding_1, Embedding_2]:
    if _url.strip(): _asset_queue.append((_url.strip(), str(Embeddings), None))
for _url in [Upscaler_1, Upscaler_2]:
    if _url.strip(): _asset_queue.append((_url.strip(), str(Upscalers), None))

if _asset_queue:
    parallel_batch_download(_asset_queue, max_workers=3)

## ⚡ FLUX Models *(Optional)*
<span style="font-size:13px;">
Download FLUX.1 model components. Works with <b>Forge</b>, <b>ComfyUI</b>, and <b>SwarmUI</b>.<br>
Leave all empty if you don't need FLUX.
</span>

In [ ]:
# @title ⚡ **FLUX Model Downloader** {"display-mode":"form"}
# @markdown ### Select FLUX Variant
FLUX_Variant = 'None' # @param ["None", "FLUX.1-schnell (Fast, 4-step)", "FLUX.1-dev (Quality, 20-step)"]
# @markdown > - **Schnell** = fastest inference, permissive license
# @markdown > - **Dev** = best quality, non-commercial license
# @markdown ---
# @markdown ### FLUX Component URLs
# @markdown > Pre-filled with recommended FP8 quantized versions (lower VRAM usage).
# @markdown > You can replace these with any compatible FLUX weights.
FLUX_Unet = 'https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-schnell-fp8.safetensors' # @param {type:"string"}
FLUX_Clip_L = 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors' # @param {type:"string"}
FLUX_T5XXL = 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors' # @param {type:"string"}
FLUX_VAE = 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors' # @param {type:"string"}
# @markdown ---
# @markdown > ⚠️ **Note**: FLUX models are large (10-25 GB total). Ensure you have enough
# @markdown > storage. Downloads run in parallel for speed.

from nenen88 import parallel_batch_download

_flux_queue = []

if FLUX_Variant != 'None':
    # Swap in Dev weights if user selected Dev
    _unet_url = FLUX_Unet
    if 'dev' in FLUX_Variant.lower() and 'schnell' in FLUX_Unet:
        _unet_url = FLUX_Unet.replace('schnell', 'dev')

    _flux_items = [
        (_unet_url,      str(UNET), None),
        (FLUX_Clip_L,    str(CLIP), None),
        (FLUX_T5XXL,     str(CLIP), None),
        (FLUX_VAE,       str(VAE),  'flux_ae.safetensors'),
    ]

    for _url, _dest, _fn in _flux_items:
        if _url.strip():
            _flux_queue.append((_url.strip(), _dest, _fn))

    print(f'\n⚡ Downloading {FLUX_Variant} components in parallel ({len(_flux_queue)} files)...')
    parallel_batch_download(_flux_queue, max_workers=2)
    print('\n✅ FLUX components ready. Make sure your WebUI has FLUX support enabled.')
else:
    print('  FLUX_Variant is "None" — skipping FLUX downloads.')

## 🎛️ ControlNet *(Optional)*
<span style="font-size:13px;">Downloads ControlNet model weights using the built-in widget.</span>

In [ ]:
# @title 🎛️ **ControlNet Widget**
%run $Controlnet_Widget

## 💨 Temporary Models
<span style="font-size:13px;">
These models are stored in <code>/tmp</code> — they <b>do not persist</b> across sessions.<br>
Use this for large test models you don't want to keep permanently.
</span>

In [ ]:
# @title 💨 **Temporary Model Downloader** {"display-mode":"form"}
# @markdown ### Temporary Checkpoints *(cleared on session end)*
# @markdown > Example: `https://civitai.com/api/download/models/357609 Juggernaut-XL_V9-RDPhoto2-Lightning_4S.safetensors`
# @markdown > Format: `URL optional_filename`
TMP_Checkpoint_1 = '' # @param {type:"string", placeholder:"URL [optional_filename] or leave empty"}
TMP_Checkpoint_2 = '' # @param {type:"string", placeholder:"URL [optional_filename] or leave empty"}
# @markdown ---
# @markdown ### Temporary LoRA *(cleared on session end)*
# @markdown > Example: `https://huggingface.co/Linaqruf/style-enhancer-xl-lora/resolve/main/style-enhancer-xl.safetensors`
TMP_Lora_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
TMP_Lora_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}

# ─── Build and run temporary download queue ───────────────────────────────────
from nenen88 import parallel_batch_download

_tmp_queue = []

for _raw in [TMP_Checkpoint_1, TMP_Checkpoint_2]:
    if not _raw.strip(): continue
    _parts = _raw.strip().split(None, 1)
    _url = _parts[0]
    _fn  = _parts[1] if len(_parts) > 1 else None
    _tmp_queue.append((_url, str(TMP_CKPT), _fn))

for _raw in [TMP_Lora_1, TMP_Lora_2]:
    if not _raw.strip(): continue
    _parts = _raw.strip().split(None, 1)
    _url = _parts[0]
    _fn  = _parts[1] if len(_parts) > 1 else None
    _tmp_queue.append((_url, str(TMP_LORA), _fn))

if not _tmp_queue:
    print('  No temporary URLs provided — skipping.')
else:
    parallel_batch_download(_tmp_queue, max_workers=3)

# 🚀 Launch
<span style="font-size:14px;">
• Add <span style="color:red;">--skip-widget</span> to load previous widget config and launch immediately.<br>
• For ComfyUI, add <span style="color:red;">--skip-comfyui-check</span> to skip dependency checks.
</span>

In [ ]:
# @title 🚀 **Launch WebUI** {"display-mode":"form"}
# @markdown ### Launch Arguments
Skip_Widget = False # @param {type:"boolean"}
# @markdown > Check to skip the launch widget and use last saved settings.
Skip_ComfyUI_Check = False # @param {type:"boolean"}
# @markdown > Check to skip ComfyUI requirement and custom node checks (faster cold start).

_args = ''
if Skip_Widget: _args += ' --skip-widget'
if Skip_ComfyUI_Check: _args += ' --skip-comfyui-check'

%cd -q $WebUI
%run segsmaker.py $_args

## 🧰 Extras

In [ ]:
# @title 🔐 **Register ZROK Account** *(one-time setup)*
%zrok_register

In [ ]:
# @title 🔑 **Change Civitai API Key**
%change_key

#### 💾 Storage Management

In [ ]:
# @title 📊 **Check Storage**
%storage

In [ ]:
# @title 🗑️ **Clear Output Images**
%clear_output_images

In [ ]:
# @title ❌ **Uninstall WebUI**
%uninstall_webui

#### 📦 Zip Output Images

In [ ]:
# @title 📦 **Zip Output Images** {"display-mode":"form"}
# @markdown Enter a name for your zip file (no extension needed).
Zip_Name = 'my_outputs' # @param {type:"string", placeholder:"e.g. my_outputs_2024"}
# @markdown Output zip will be saved to your HOME directory.

%%zipping
name    = Zip_Name
inputs  = $WebUI_Output
outputs = $HOME